# Model D-Pro Phase 1 — reusable package

This notebook imports the migrated modules. Historical pilot: **21.1574 dB / 0.736978 SSIM**, best update **900**, 1,200 updates completed. These are archived measurements, not measurements made by this notebook. No D-Pro Test score has been measured; 23.1 dB remains an unverified target.

**Training and full validation are disabled by default.** Start with a smoke check. `Run all` cannot start training unless you explicitly enable it below. Model weights and results stay on Drive. Select a T4 for the original training recipe; CPU supports smoke/parity checks.


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')
SCREENING = DRIVE_ROOT / 'RetinexFormer_NAF_Screening'
DATA_ROOT = DRIVE_ROOT / 'datasets/LOLv2/Real_captured'
MODEL_B = SCREENING / 'model_B_optionA_batch8_patch128_lr1e-5_from_batch8best/low_lr_1e-5_1000steps/best.pth'
HEAD = SCREENING / 'model_D_HVI_from_ModelB_OptionA_v1/best_model_d_head.pth'
PHASE1_BEST = SCREENING / 'model_DPro_budget_v1/pilot_01/best.pth'
SPLIT = SCREENING / 'model_B_v1_full_data_1000steps/split.json'
OUTPUT_ROOT = SCREENING / 'model_DPro_modular_phase1'
RUN_NAME = 'phase1_01'  # Never use the historical pilot_01 directory.
MODEL_SOURCE = 'phase1'  # 'phase1' or 'original'; training always uses the original pair.
DEVICE = 'cuda'
RUN_SMOKE = True
RUN_VERIFY = False
RUN_VALIDATION = False
RUN_TRAINING = False
COMPARE_HISTORICAL = True  # Applies only to validation of the step-900 checkpoint.
RESUME = False  # Modular runs only. Historical last.pth is not a full-state resume input.
MAX_STEPS = 1200
VALIDATE_EVERY = 300
SAVE_EVERY = 100
SESSION_MINUTES = 45
AUTO_DISCONNECT = False  # Explicit choice; invoked only after requested work ran.
REPO_URL = 'https://github.com/Silly-Gooses/myRetinexformer.git'
REPO = Path('/content/myRetinexformer_model_d_pro')


## Setup
Mount Drive, clone or fast-forward `model_d_pro`, and install only missing small dependencies. A dirty or diverged checkout is rejected. CUDA PyTorch is left in place.


In [ ]:
import importlib.util
import json
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')
def checked(command, **kwargs):
    return subprocess.run(command, check=True, **kwargs)

if not (REPO / '.git').is_dir():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError('Nonempty non-git checkout directory')
    checked(['git', 'clone', '--branch', 'model_d_pro', '--single-branch', REPO_URL, str(REPO)])
else:
    remote = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], cwd=REPO, text=True).strip()
    if remote.rstrip('/') != REPO_URL:
        raise RuntimeError('Unexpected repository remote')
    if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
        raise RuntimeError('Checkout is dirty; use a fresh checkout')
    checked(['git', 'fetch', 'origin', 'model_d_pro'], cwd=REPO)
    checked(['git', 'checkout', 'model_d_pro'], cwd=REPO)
    checked(['git', 'merge', '--ff-only', 'origin/model_d_pro'], cwd=REPO)
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('Use a Colab runtime with PyTorch installed')
needed = [package for module, package in [('einops', 'einops>=0.7,<1'),
          ('cv2', 'opencv-python-headless>=4.8,<5'), ('PIL', 'Pillow>=9,<13')]
          if importlib.util.find_spec(module) is None]
if needed:
    checked([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', *needed])
sys.path.insert(0, str(REPO))
from model_d_pro import ModelDPro, load_model_d_pro
from model_d_pro.cli import DEFAULT_CONFIG
from model_d_pro.checkpoint import source_manifest
from model_d_pro.train import protect_output, validate_config
from model_d.model import REFERENCE
source_manifest(REPO)
print('Using commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())


## Check inputs and configure outputs
Choose either the original trained pair or Phase 1 best weights for smoke, inference, and validation. Full validation always uses the frozen 69-image split. A missing optional split file falls back to the identical repository split.


In [ ]:
from datetime import datetime, timezone

if MODEL_SOURCE not in {'phase1', 'original'}:
    raise ValueError('MODEL_SOURCE must be phase1 or original')
required = [PHASE1_BEST] if MODEL_SOURCE == 'phase1' else [MODEL_B, HEAD]
if RUN_TRAINING:
    required += [MODEL_B, HEAD]
for path in required:
    if not path.is_file():
        raise FileNotFoundError(path)
for part in ['Train/Low', 'Train/Normal']:
    if not (DATA_ROOT / part).is_dir():
        raise FileNotFoundError(DATA_ROOT / part)
CFG = json.loads(DEFAULT_CONFIG.read_text())
CFG.update(data_root=str(DATA_ROOT), model_b=str(MODEL_B), head=str(HEAD),
           split_file=str(SPLIT), output_root=str(OUTPUT_ROOT), run_name=RUN_NAME,
           resume=RESUME, max_steps=MAX_STEPS, validate_every=VALIDATE_EVERY, save_every=SAVE_EVERY)
validate_config(CFG, SESSION_MINUTES)
protect_output(OUTPUT_ROOT / RUN_NAME, [DATA_ROOT, MODEL_B, HEAD, PHASE1_BEST])
CONFIG_PATH = Path('/content/model_dpro_phase1_config.json')
CONFIG_PATH.write_text(json.dumps(CFG, indent=2))
SESSION_TAG = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
REPORT_ROOT = OUTPUT_ROOT / 'checks' / SESSION_TAG
SMOKE_IMAGE = DATA_ROOT / 'Train/Low' / REFERENCE['config']['train_names'][0]
VERIFY_IMAGE = DATA_ROOT / 'Train/Low' / REFERENCE['config']['val_names'][0]
weights = ['--checkpoint', str(PHASE1_BEST)] if MODEL_SOURCE == 'phase1' else ['--model-b', str(MODEL_B), '--head', str(HEAD)]
print('Check reports:', REPORT_ROOT)
print('Training output:', OUTPUT_ROOT / RUN_NAME)
print('Training enabled:', RUN_TRAINING, '| update cap:', MAX_STEPS, '| session minutes:', SESSION_MINUTES)


## Run selected work
Smoke performs forward/backward checks without an optimizer update. Verification compares identical weights and inputs against the archived script. Historical metric comparison requires the step-900 best checkpoint. Training uses the exact original initialization and recipe; it never starts Phase 2. The worker enforces the update cap, saving reserve, and outer watchdog.


In [ ]:
work_started = False
try:
    for command, enabled, extra in [
        ('smoke', RUN_SMOKE, ['--input', str(SMOKE_IMAGE)]),
        ('verify', RUN_VERIFY, ['--input', str(VERIFY_IMAGE)]),
        ('evaluate', RUN_VALIDATION, ['--session-minutes', str(SESSION_MINUTES)] +
         (['--compare-historical'] if COMPARE_HISTORICAL and MODEL_SOURCE == 'phase1' else [])),
    ]:
        if enabled:
            work_started = True
            checked([sys.executable, '-m', 'model_d_pro.cli', command, '--config', str(CONFIG_PATH),
                     '--device', DEVICE, '--output-dir', str(REPORT_ROOT / command), *weights, *extra], cwd=REPO)
    if RUN_TRAINING:
        work_started = True
        checked([sys.executable, '-m', 'model_d_pro.cli', 'train', '--config', str(CONFIG_PATH),
                 '--session-minutes', str(SESSION_MINUTES)], cwd=REPO)
    else:
        print('Training is disabled. No optimizer updates requested.')
finally:
    if AUTO_DISCONNECT and work_started:
        from google.colab import runtime
        runtime.unassign()


## Results and future work
Read the newly written `report.json`, `parity.json`, and validation CSV on Drive. A nonzero command exit means a check failed; inspect the saved report before claiming reproduction. Synthetic CPU tests cannot establish the historical trained score.

Historical `best.pth` and `last.pth` are accepted for **model weights only**. Full-state resume is supported only within compatible modular runs (`RESUME=True`, unchanged source/recipe/data, same run directory). A completed modular run does not add more updates. A changed recipe needs a new run name.

For future Phase 2, use `load_model_d_pro(checkpoint=PHASE1_BEST, device=DEVICE)` and define a separate optimizer and schedule. This notebook provides no Phase 2 training recipe and never evaluates Test.
